# Automatizando o treinamento com `make_pipeline`

Durante o desenvolvimento de modelos de aprendizado de máquina, é comum combinar etapas de preparação e modelagem. Por exemplo, podemos escalonar os dados, reduzir sua dimensionalidade com PCA e, por fim, treinar um classificador.

O `make_pipeline`, do scikit-learn, encadeia essas etapas em um único objeto. Isso organiza o código e ajuda a executar corretamente as transformações dentro da validação cruzada.

Neste notebook, vamos comparar combinações de escaladores, PCA e classificadores usando o conjunto de dados **Wine**.

## Por que usar um pipeline?

Sem um pipeline, precisaríamos ajustar cada transformação separadamente em cada divisão de treino e validação, aumentando o risco de erros e de **vazamento de dados** (*data leakage*).

O fluxo deste exemplo será:

**Escalonador → PCA → Classificador**

Durante a validação cruzada, cada etapa é ajustada somente com a parte de treino daquela divisão. As transformações aprendidas são então aplicadas à parte de validação.

## Carregando o conjunto de dados

O conjunto Wine contém características químicas de vinhos e um rótulo de classe. Vamos usar as características para prever a classe e comparar os modelos com validação cruzada.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.datasets import load_wine
from sklearn.pipeline import make_pipeline

dataset = load_wine()
X = dataset.data
y = dataset.target

print("Formato das características:", X.shape)
print("Classes:", np.unique(y))

## Definindo os classificadores

Vamos avaliar três algoritmos:

- **SVC**: classificador baseado em máquinas de vetores de suporte.
- **KNeighborsClassifier (KNC)**: classifica observações a partir das classes dos vizinhos mais próximos.
- **SGDClassifier (SGDC)**: treina um classificador usando descida de gradiente estocástica.

Usaremos as configurações padrão dos modelos, com uma semente fixa para o `SGDClassifier` para facilitar a reprodução do experimento.

In [ ]:
models = [
    {"model": SVC(), "name": "SVC"},
    {"model": KNeighborsClassifier(), "name": "KNC"},
    {"model": SGDClassifier(random_state=42), "name": "SGDC"}
]

## Definindo os escaladores

Vamos comparar duas técnicas:

- **MinMaxScaler**: transforma cada característica para o intervalo padrão entre 0 e 1.
- **StandardScaler**: centraliza cada característica e a divide pelo desvio padrão.

O escalonamento antes do PCA é importante porque o PCA é sensível à variância. Sem escalonamento, características com magnitudes maiores podem dominar os componentes principais.

In [ ]:
scalers = [
    {"scaler": MinMaxScaler(), "name": "MinMax"},
    {"scaler": StandardScaler(), "name": "Standard"}
]

## Definindo o PCA

O PCA (*Principal Component Analysis*) representa os dados em um conjunto menor de componentes, buscando preservar a variância.

Quando `n_components` é um número entre 0 e 1, ele indica a proporção de variância que se deseja preservar. Por exemplo, `PCA(n_components=0.95)` seleciona componentes suficientes para explicar aproximadamente 95% da variância observada nos dados usados para ajustar o PCA.

Vamos testar quatro valores: 95%, 88%, 66% e 50%. O número de componentes resultante pode variar entre escaladores e divisões da validação cruzada.

In [ ]:
feat_extrs = [
    {"pca": PCA(n_components=0.95), "name": "95%"},
    {"pca": PCA(n_components=0.88), "name": "88%"},
    {"pca": PCA(n_components=0.66), "name": "66%"},
    {"pca": PCA(n_components=0.50), "name": "50%"}
]

## Configurando a validação cruzada

Usaremos `StratifiedKFold` com cinco divisões. A estratificação procura manter proporções de classes semelhantes em cada divisão. `shuffle=True` embaralha as observações, e `random_state=42` torna a separação reproduzível.

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Montando e avaliando os pipelines

A função `make_pipeline` recebe as etapas na ordem de execução e cria um pipeline. Cada combinação será:

**Escalador → PCA → Classificador**

A função `cross_validate` avalia o pipeline nas cinco divisões. Vamos calcular:

- **Accuracy**: proporção de previsões corretas.
- **F1 macro**: calcula o F1 para cada classe e depois tira a média sem ponderar pela quantidade de observações de cada classe.

Também guardaremos os resultados em uma tabela para facilitar a comparação.

In [ ]:
resultados = []

for scaler_info in scalers:
    for pca_info in feat_extrs:
        for model_info in models:
            print(
                f"Scaler: {scaler_info['name']} - "
                f"PCA: {pca_info['name']} - "
                f"Modelo: {model_info['name']}"
            )

            pipeline = make_pipeline(
                scaler_info["scaler"],
                pca_info["pca"],
                model_info["model"]
            )

            scores = cross_validate(
                pipeline,
                X,
                y,
                cv=kf,
                scoring={
                    "accuracy": "accuracy",
                    "f1_macro": "f1_macro"
                }
            )

            accuracy_mean = np.mean(scores["test_accuracy"])
            f1_mean = np.mean(scores["test_f1_macro"])

            print(f"Accuracy média: {accuracy_mean:.4f}")
            print(f"F1 macro médio: {f1_mean:.4f}\n")

            resultados.append({
                "Scaler": scaler_info["name"],
                "PCA (variância)": pca_info["name"],
                "Modelo": model_info["name"],
                "Accuracy média": accuracy_mean,
                "F1 macro médio": f1_mean
            })

resultados_df = pd.DataFrame(resultados)
resultados_df.head()

## Comparando os resultados

A tabela contém todas as combinações avaliadas. Vamos ordená-la pelo F1 macro para observar quais combinações tiveram os maiores valores médios neste experimento.

A melhor combinação nesta execução não é necessariamente a melhor para qualquer conjunto de dados: o resultado depende dos dados, das configurações e da estratégia de validação.

In [ ]:
resultados_df.sort_values(
    by="F1 macro médio",
    ascending=False
).reset_index(drop=True)

Também podemos agrupar os resultados por classificador. Isso ajuda a explorar tendências, mas não substitui uma análise cuidadosa da variação entre as divisões.

In [ ]:
(
    resultados_df
    .groupby("Modelo")[["Accuracy média", "F1 macro médio"]]
    .mean()
    .sort_values("F1 macro médio", ascending=False)
)

## Conceitos importantes

- **Pipeline:** encadeia transformações e estimador em um único objeto.
- **`make_pipeline`:** cria um pipeline e gera automaticamente nomes para as etapas.
- **PCA:** reduz a dimensionalidade, mantendo uma proporção escolhida da variância.
- **Validação cruzada estratificada:** avalia o processo em diferentes divisões, mantendo as proporções de classes tão equilibradas quanto possível.
- **Accuracy:** mede a proporção total de previsões corretas.
- **F1 macro:** dá o mesmo peso ao F1 de cada classe.

Como o escalador e o PCA estão dentro do pipeline, cada divisão da validação cruzada aprende as transformações somente com os dados de treino daquela divisão, ajudando a evitar vazamento de dados.